# Monte Carlo estimation of pi

Each worker throws a few million random darts at the unit square and counts how many land inside the unit circle. The client sums the per-worker counts and recovers pi.

This is the canonical embarrassingly-parallel demo: every task is independent, the inputs are tiny (a seed and a sample count), and the result is a single integer -- so almost all of the wall-clock time is real numerical work done on the workers.

**Prerequisites**: a Scaler scheduler reachable from this kernel. See :doc:`../tutorials/try_in_browser` for the recommended worker setup.

In [ ]:
# Connection settings -- edit these to point at your running cluster.
SCHEDULER_ADDRESS = "ws://127.0.0.1:2345"  # supports tcp:// or ws://; only ws:// works from JupyterLite (browser)
OBJECT_STORAGE_ADDRESS = None  # leave None to use whatever the scheduler advertises

N_TASKS = 64
SAMPLES_PER_TASK = 2_000_000  # ~16 MB of random floats per task -- entirely worker-side

In [ ]:
import time

from scaler import Client


def count_inside_unit_circle(seed: int, n: int) -> int:
    """Worker-side: throw `n` random darts, return how many landed inside the unit circle."""
    import numpy as np

    rng = np.random.default_rng(seed)
    x = rng.random(n, dtype=np.float64)
    y = rng.random(n, dtype=np.float64)
    return int(np.count_nonzero(x * x + y * y <= 1.0))


with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    started = time.perf_counter()
    futures = [client.submit(count_inside_unit_circle, seed, SAMPLES_PER_TASK) for seed in range(N_TASKS)]
    inside = sum(future.result() for future in futures)
    elapsed = time.perf_counter() - started

total = N_TASKS * SAMPLES_PER_TASK
pi_estimate = 4.0 * inside / total
print(f"threw {total:,} darts across {N_TASKS} tasks in {elapsed:.2f}s")
print(f"pi ~= {pi_estimate:.6f}  (error {abs(pi_estimate - 3.141592653589793):.2e})")